# Reader note

This notebook is part of the `arXiv:2606.04091` reproduction workflow. It reproduces upper panel of `Figures 6, 7,and 8`, the daily modulation amplitute for different materials calculated as Eq(4.2):
$$f_{mod} = max|R(t)-<R>|/<R>$$
Set `DATA_ROOT` to the directory containing the generated HDF5 data. Old execution outputs are intentionally cleared for release.

In [ ]:
import os
import h5py
import script_helpers.script_fmod as dm
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np

# Generated HDF5 data root. Override this without editing the notebook by setting
DATA_ROOT='/path/to/generated/data before starting Jupyter.'

In [ ]:
### Set the target materials to plot (CaWO4 SiO2, Al2O3)
Targets = ["Al2O3", "SiO2", "CaWO4"]

### Set the threshold value to plot
new_threshold = 0.001 # 1 meV, 20 meV etc. (in eV)

### Define mediator types to plot (light_hadrophilic, heavy_hadrophilic, light_dark_photon)
Mediators = ["light_dark_photon"] # Choose the mediator name you want to plot

### numerics used in calculations (standard, standard_q_parallel)
Numerics = "standard"

### Halo models used in calculations (SHM, TSA, EMP)
Mod = ["SHM", "TSA", "EMP"]

In [ ]:
### Conservative Velocity parameters in standard prescription
V_0 = [220, 200, 280]
V_E = [232, 217, 246]
V_ESC = [544, 450, 600]
fid_vel = "220_232_544" # Central values of the velocity parameters in the standard prescription

### V_0 parameters for TSA and EMP in rms matching prescription
V_0_tsa_h = [268, 234, 381] # vesc =600
V_0_tsa_m = [280, 244, 398] # vesc =544
V_0_tsa_l = [306, 266, 430] # vesc =450
fid_velocity_tsa = "280_232_544"

V_0_emp_h = [100, 126, 326] # vesc =600
V_0_emp_m = [115, 154, 710] # vesc =544
V_0_emp_l = [187, 340, np.inf] # vesc =450
fid_velocity_emp = "154_232_544"

V_ESC_h= [600]
V_ESC_m= [544]
V_ESC_l= [450]

In [ ]:
### Function to get the data directory for a given target and subdirectory
def get_data_dir(Target, subdir):
    """
    subdir = 'files' or 'files_new'
    """
    # Change this to the path where you have the generated HDF5 files for daily modulation
    # Current path is set to the publicly available data repository for the paper
    return DATA_ROOT + f"/Manuscript_data/{Target}_daily_modulation/{subdir}"

In [ ]:
### Build fmod results for a given subdirectory
def build_fmod_results_for_subdir(subdir):
    fmod_results = {}

    for Target in Targets:
        data_dir = get_data_dir(Target, subdir)

        for mediator_key in Mediators:

            # Generate file prefixes
            prefixes_SHM = dm.generate_file_prefixes(Target, [mediator_key], Numerics, Mod[0], V_0, V_E, V_ESC)
            prefixes_TSA = dm.generate_file_prefixes(Target, [mediator_key], Numerics, Mod[1], V_0, V_E, V_ESC)
            prefixes_EMP = dm.generate_file_prefixes(Target, [mediator_key], Numerics, Mod[2], V_0, V_E, V_ESC)

            # Load scan files
            files_SHM = [h5py.File(os.path.join(data_dir, p + ".hdf5"), "r") for p in prefixes_SHM]
            files_TSA = [h5py.File(os.path.join(data_dir, p + ".hdf5"), "r") for p in prefixes_TSA]
            files_EMP = [h5py.File(os.path.join(data_dir, p + ".hdf5"), "r") for p in prefixes_EMP]

            # Fiducial filenames
            fileSHM = f"{Target}_{mediator_key}_standard_{Mod[0]}_{fid_vel}.hdf5"
            fileTSA = f"{Target}_{mediator_key}_standard_{Mod[1]}_{fid_vel}.hdf5"
            fileEMP = f"{Target}_{mediator_key}_standard_{Mod[2]}_{fid_vel}.hdf5"

            fileSHM_path = os.path.join(data_dir, fileSHM)
            fileTSA_path = os.path.join(data_dir, fileTSA)
            fileEMP_path = os.path.join(data_dir, fileEMP)

            # Shared grids
            DM_mass = files_SHM[0]['particle_physics/dm_properties/mass_list'][()] * 1e-6
            time = files_SHM[0]['particle_physics/times'][()]
            Threshold = files_SHM[0]['particle_physics/threshold'][()]
            energy_bin_width = files_SHM[0]['numerics/energy_bin_width'][()]

            if len(Threshold) == 1:
                th = 0
            else:
                print(f'{Target} - {mediator_key}: Pick threshold from {Threshold}')
                th = 0  # or handle interactively

            threshold_run = Threshold[th]

            # Max/min envelopes
            fmods_max_SHM, fmods_min_SHM = dm.calculate_fmod_maxmin_from_diff(
                files_SHM, time, DM_mass, th, new_threshold, threshold_run, energy_bin_width
            )
            fmods_max_TSA, fmods_min_TSA = dm.calculate_fmod_maxmin_from_diff(
                files_TSA, time, DM_mass, th, new_threshold, threshold_run, energy_bin_width
            )
            fmods_max_EMP, fmods_min_EMP = dm.calculate_fmod_maxmin_from_diff(
                files_EMP, time, DM_mass, th, new_threshold, threshold_run, energy_bin_width
            )

            # Fiducial curves
            with h5py.File(fileSHM_path, "r") as f_shm:
                fmod_SHM = dm.calculate_fmod_from_diff(
                    f_shm, time, DM_mass, th, new_threshold, threshold_run, energy_bin_width
                )

            with h5py.File(fileTSA_path, "r") as f_tsa:
                fmod_TSA = dm.calculate_fmod_from_diff(
                    f_tsa, time, DM_mass, th, new_threshold, threshold_run, energy_bin_width
                )

            with h5py.File(fileEMP_path, "r") as f_emp:
                fmod_EMP = dm.calculate_fmod_from_diff(
                    f_emp, time, DM_mass, th, new_threshold, threshold_run, energy_bin_width
                )

            # Store
            fmod_results[(Target, mediator_key, "SHM")] = {
                "fmod_max": fmods_max_SHM,
                "fmod_min": fmods_min_SHM,
                "fiducial": fmod_SHM,
                "DM_mass": DM_mass,
                "time": time,
                "threshold_index": th,
            }
            fmod_results[(Target, mediator_key, "TSA")] = {
                "fmod_max": fmods_max_TSA,
                "fmod_min": fmods_min_TSA,
                "fiducial": fmod_TSA,
                "DM_mass": DM_mass,
                "time": time,
                "threshold_index": th,
            }
            fmod_results[(Target, mediator_key, "EMP")] = {
                "fmod_max": fmods_max_EMP,
                "fmod_min": fmods_min_EMP,
                "fiducial": fmod_EMP,
                "DM_mass": DM_mass,
                "time": time,
                "threshold_index": th,
            }

            # Close scan files
            for f in files_SHM + files_TSA + files_EMP:
                f.close()

    return fmod_results

In [ ]:
### Build fmod results in rms matching prescription for a given subdirectory
def build_fmod_results_for_subdir_rms(subdir):
    fmod_results = {}

    for Target in Targets:
        data_dir = get_data_dir(Target, subdir)

        for mediator_key in Mediators:

            # Generate file prefixes
            # prefixes_SHM = dm.generate_file_prefixes(Target, [mediator_key], Numerics, Mod[0], V_0, V_E, V_ESC)

            prefixes_tsa_l = dm.generate_file_prefixes(Target, [mediator_key], Numerics, Mod[1], V_0_tsa_l, V_E, V_ESC_l)
            prefixes_tsa_m = dm.generate_file_prefixes(Target, [mediator_key], Numerics, Mod[1], V_0_tsa_m, V_E, V_ESC_m)
            prefixes_tsa_h = dm.generate_file_prefixes(Target, [mediator_key], Numerics, Mod[1], V_0_tsa_h, V_E, V_ESC_h)
            # prefixes_tsa = defaultdict(list)
            # for d in [prefixes_tsa_h, prefixes_tsa_m, prefixes_tsa_l]:
            #     for key, values in d.items():
            #         prefixes_tsa[key].extend(values)
            prefixes_tsa = prefixes_tsa_h + prefixes_tsa_m + prefixes_tsa_l

            prefixes_emp_l = dm.generate_file_prefixes(Target, [mediator_key], Numerics, Mod[2], V_0_emp_l, V_E, V_ESC_l)
            prefixes_emp_m = dm.generate_file_prefixes(Target, [mediator_key], Numerics, Mod[2], V_0_emp_m, V_E, V_ESC_m)
            prefixes_emp_h = dm.generate_file_prefixes(Target, [mediator_key], Numerics, Mod[2], V_0_emp_h, V_E, V_ESC_h)
            # prefixes_emp = defaultdict(list)
            # for d in [prefixes_emp_h, prefixes_emp_m, prefixes_emp_l]:
            #     for key, values in d.items():
            #         prefixes_emp[key].extend(values)
            prefixes_emp = prefixes_emp_h + prefixes_emp_m + prefixes_emp_l

            # Load scan files
            # files_SHM = [h5py.File(os.path.join(data_dir, p + ".hdf5"), "r") for p in prefixes_SHM]

            files_tsa = [h5py.File(os.path.join(data_dir, p + ".hdf5"), "r") for p in prefixes_tsa]
            files_emp = [h5py.File(os.path.join(data_dir, p + ".hdf5"), "r") for p in prefixes_emp]

            # Fiducial filenames
            # fileSHM = f"{Target}_{mediator_key}_standard_{Mod[0]}_{fid_vel}.hdf5"

            filetsa = f"{Target}_{mediator_key}_standard_{Mod[1]}_{fid_velocity_tsa}.hdf5"
            fileemp = f"{Target}_{mediator_key}_standard_{Mod[2]}_{fid_velocity_emp}.hdf5"

            # fileSHM_path = os.path.join(data_dir, fileSHM)

            filetsa_path = os.path.join(data_dir, filetsa)
            fileemp_path = os.path.join(data_dir, fileemp)

            # Shared grids
            DM_mass = files_tsa[0]['particle_physics/dm_properties/mass_list'][()] * 1e-6
            time = files_tsa[0]['particle_physics/times'][()]
            Threshold = files_tsa[0]['particle_physics/threshold'][()]
            energy_bin_width = files_tsa[0]['numerics/energy_bin_width'][()]

            if len(Threshold) == 1:
                th = 0
            else:
                print(f'{Target} - {mediator_key}: Pick threshold from {Threshold}')
                th = 0  # or handle interactively

            threshold_run = Threshold[th]

            # Max/min envelopes
            # fmods_max_SHM, fmods_min_SHM = dm.calculate_fmod_maxmin_from_diff(
            #     files_SHM, time, DM_mass, th, new_threshold, threshold_run, energy_bin_width
            # )

            fmods_max_tsa, fmods_min_tsa = dm.calculate_fmod_maxmin_from_diff(
                files_tsa, time, DM_mass, th, new_threshold, threshold_run, energy_bin_width
            )
            fmods_max_emp, fmods_min_emp = dm.calculate_fmod_maxmin_from_diff(
                files_emp, time, DM_mass, th, new_threshold, threshold_run, energy_bin_width
            )

            # Fiducial curves
            # with h5py.File(fileSHM_path, "r") as f_shm:
            #     fmod_SHM = dm.calculate_fmod_from_diff(
            #         f_shm, time, DM_mass, th, new_threshold, threshold_run, energy_bin_width
            #     )

            with h5py.File(filetsa_path, "r") as f_tsa1:
                fmod_tsa = dm.calculate_fmod_from_diff(
                    f_tsa1, time, DM_mass, th, new_threshold, threshold_run, energy_bin_width
                )

            with h5py.File(fileemp_path, "r") as f_emp1:
                fmod_emp = dm.calculate_fmod_from_diff(
                    f_emp1, time, DM_mass, th, new_threshold, threshold_run, energy_bin_width
                )

            # # Store
            # fmod_results[(Target, mediator_key, "SHM")] = {
            #     "fmod_max": fmods_max_SHM,
            #     "fmod_min": fmods_min_SHM,
            #     "fiducial": fmod_SHM,
            #     "DM_mass": DM_mass,
            #     "time": time,
            #     "threshold_index": th,
            # }
            fmod_results[(Target, mediator_key, "TSA")] = {
                "fmod_max_rms": fmods_max_tsa,
                "fmod_min_rms": fmods_min_tsa,
                "fiducial_rms": fmod_tsa,
                "DM_mass": DM_mass,
                "time": time,
                "threshold_index": th,
            }
            fmod_results[(Target, mediator_key, "EMP")] = {
                "fmod_max_rms": fmods_max_emp,
                "fmod_min_rms": fmods_min_emp,
                "fiducial_rms": fmod_emp,
                "DM_mass": DM_mass,
                "time": time,
                "threshold_index": th,
            }

            # Close scan files
            for f in files_tsa + files_emp:
                f.close()

    return fmod_results

In [ ]:
### Build fmod results for different subdirectories
# Change this to the path where you have the generated HDF5 files for daily modulation
# Current path is set to the publicly available data repository for the paper
fmod_results = build_fmod_results_for_subdir("files")
fmod_results_2 = build_fmod_results_for_subdir("files_QQQ_10-3")
fmod_results_rms = build_fmod_results_for_subdir_rms("files_vrms_QQQ")

# For the standard SHM prescription, the data were generated in two separate
# mass ranges, so fmod_results and fmod_results_2 are combined to cover the
# full mass range shown in the final plot.
# For the RMS-matching prescription (tSA and EMP), the full mass range was
# generated in a single run and therefore does not require stitching.

In [ ]:
# Setup
model_colors = {"SHM": "#56B4E9", "TSA": "#E69F00", "EMP": "#CC79A7"}
target_colors = {"CaWO4": "green", "SiO2": "darkturquoise", "Al2O3": "magenta"}
linestyles_new = {"max": "dashed", "min": "dotted"}

## Making the matplotlib plots look nicer
settings = {
    # 'figure.constrained_layout.use': True,
    # 'mathtext.fontset': 'stix',
    # 'font.family': 'STIXGeneral',
    # LaTeX-like fonts
    'mathtext.fontset': 'cm',
    # 'font.family': 'serif',
    # 'font.serif': ['Computer Modern Roman'],
    # 'mathtext.fontset': 'dejavuserif',
    'font.family': 'DejaVu Serif',
    'font.size':13,
    # 'axes.labelsize': 'large',
    'lines.markersize': 5,
    'axes.linewidth':2.0,
    'xtick.major.size':8.0,
    'xtick.minor.size':4.0,
    'xtick.major.width':1.5,
    'xtick.minor.width':1.0,
    'xtick.direction':'in', 
    'xtick.minor.visible':True,
    'xtick.top':True,
    'ytick.major.size':8.0,
    'ytick.minor.size':4.0,
    'ytick.major.width':1.5,
    'ytick.minor.width':1.0,
    'ytick.direction':'in', 
    'ytick.minor.visible':True,
    'ytick.right':True,
    'contour.linewidth':3.0,
    'savefig.bbox': 'tight',
    'savefig.dpi': 200,
}

plt.rcParams.update(**settings) 

In [ ]:
# Select the mediator you want to plot
mediator_to_plot = Mediators[0]  # Choose the mediator name you want to plot if you list multiple mediators in the Mediators list. 

In [ ]:
### Plotting fmod bands for all Targets and one chosen mediator

plt.figure(figsize=(8, 6))

for Target in Targets:
    key_SHM = (Target, mediator_to_plot, 'SHM')
    SHM_data = fmod_results[key_SHM]
    DM_mass = SHM_data['DM_mass']

    SHM_data_2 = fmod_results_2[key_SHM]
    DM_mass_2 = SHM_data_2['DM_mass']

    key_TSA = (Target, mediator_to_plot, 'TSA')
    TSA_data = fmod_results_rms.get(key_TSA, None)
    DM_mass_rms = TSA_data['DM_mass']

    key_EMP = (Target, mediator_to_plot, 'EMP')
    EMP_data = fmod_results_rms.get(key_EMP, None)

    # SHM band
    plt.fill_between(DM_mass, SHM_data['fmod_min'], SHM_data['fmod_max'],
                     color= target_colors[Target], alpha=0.15, step="pre")
    plt.fill_between(DM_mass_2, SHM_data_2['fmod_min'], SHM_data_2['fmod_max'],
                     color= target_colors[Target], alpha=0.15, step="pre")

    # SHM curve
    plt.step(DM_mass,
             SHM_data['fiducial'],
             where="pre",
             color=target_colors[Target],
             alpha=0.6,              
             linewidth=1.5 ,
             label=f'{Target} fiducial')
    plt.step(DM_mass_2,
             SHM_data_2['fiducial'],
             where="pre",
             color=target_colors[Target],
             alpha=0.6,              
             linewidth=1.5 ,
             label=f'{Target} fiducial')

    
    # TSA curve
    if TSA_data is not None:
        plt.step(DM_mass_rms,
             TSA_data['fiducial_rms'],
             where="pre",
             color=target_colors[Target],
             alpha=0.6,              
             linewidth=1.0 ,
             linestyle=linestyles_new["max"])
        
    # EMP curve
    if EMP_data is not None:
        plt.step(DM_mass_rms,
                EMP_data['fiducial_rms'],
                where="pre",
                color=target_colors[Target],
                alpha=0.6,              
                linewidth=1.0 ,
                linestyle=linestyles_new["min"])
    
# Formatting 
plt.xlabel(r"$m_\chi$ (MeV)", fontsize=22)
plt.ylabel(rf"$f_{{\mathrm{{mod}}}}$", fontsize=22)
plt.xscale("log")
plt.yscale("log")
plt.xlim(3*1e-3, 1e0) 
plt.ylim(1e-2, 1e1)

handles = [
    Line2D([0], [0], color=target_colors["CaWO4"], lw=1.5, linestyle='-',  label='CaWO4'),
    Line2D([0], [0], color=target_colors["SiO2"], lw=1.5, linestyle='-',  label='SiO2'),
    Line2D([0], [0], color=target_colors["Al2O3"], lw=1.5, linestyle='-',  label='Al2O3'),
]

plt.legend(
    handles=handles,
    loc='upper right',     
    ncol=1,                
    frameon=False,
    fontsize=18
)
plt.text(0.10, 0.90, f'$\omega_{{min}}$= {new_threshold*1e3:.0f} meV', transform=plt.gca().transAxes, verticalalignment='top',
                      alpha=0.8, fontsize=20)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()